# Measuring the latency impact of prompt caching

This notebook demonstrates how to measure the latency benefit of [prompt caching](https://docs.aws.amazon.com/bedrock/latest/userguide/prompt-caching.html) on Amazon Bedrock using LLMeter.

Prompt caching allows the model to skip reprocessing a static prefix (e.g. a long system prompt) on repeated requests, reducing both **time to first token (TTFT)** and **cost**. This notebook compares TTFT distributions with and without cache hits, using a `before_invoke` callback to bust the cache on demand.

## Setup

In [1]:
# %pip install "llmeter[plotting]<1"

In [2]:
from llmeter.endpoints import BedrockConverseStream
from llmeter.runner import Runner

Choose a model that supports prompt caching. Amazon Nova models support caching with a minimum of 1024 tokens in the cached prefix. You must use an [inference profile ID](https://docs.aws.amazon.com/bedrock/latest/userguide/cross-region-inference-support.html) (e.g. `us.amazon.nova-lite-v1:0`), not the base model ID. See the [supported models table](https://docs.aws.amazon.com/bedrock/latest/userguide/prompt-caching.html) for details.

In [3]:
model_id = "us.amazon.nova-lite-v1:0"

endpoint = BedrockConverseStream(
    model_id=model_id,
    # region="us-east-1",
)

## Building a cacheable payload

Prompt caching requires:
1. A static prefix long enough to meet the model's minimum (1024 tokens for Nova)
2. A `cachePoint` marker after the content to cache

We'll use a long system prompt as the cached prefix, with a short user message that varies.

In [4]:
# Build a system prompt that exceeds the 1024-token minimum.
# Each rule line is ~25 tokens, so 80 lines ≈ 2000 tokens.
long_system_text = "You are a helpful assistant. Follow these rules:\n" + "\n".join(
    f"Rule {i}: Always be concise, accurate, and helpful in your responses. "
    "This is an important guideline that must be followed at all times."
    for i in range(1, 81)
)

payload_with_cache = {
    "system": [
        {"text": long_system_text},
        {"cachePoint": {"type": "default"}},
    ],
    "messages": [
        {
            "role": "user",
            "content": [{"text": "What is the capital of France? Answer in one word."}],
        }
    ],
    "inferenceConfig": {"maxTokens": 50},
}

Let's verify caching works with a quick test — the first call writes to cache, the second should read from it:

In [5]:
import time
from uuid import uuid4

# Use a unique prefix so re-running the notebook doesn't hit a stale cache
verify_prefix = f"[verify-{uuid4().hex}] "
verify_payload = {
    **payload_with_cache,
    "system": [
        {"text": verify_prefix + long_system_text},
        {"cachePoint": {"type": "default"}},
    ],
}

# First call: cache write (fresh prefix guarantees a miss)
r1 = endpoint.invoke(verify_payload)
print(
    f"Call 1: TTFT={r1.time_to_first_token:.3f}s, cached_tokens={r1.num_tokens_input_cached}"
)

time.sleep(1)  # Brief pause for cache propagation

# Second call: same prefix, should hit the cache
r2 = endpoint.invoke(verify_payload)
print(
    f"Call 2: TTFT={r2.time_to_first_token:.3f}s, cached_tokens={r2.num_tokens_input_cached}"
)

Call 1: TTFT=1.126s, cached_tokens=0
Call 2: TTFT=0.576s, cached_tokens=2435


## Cache-busting callback

To measure latency **without** cache hits, we need every request to miss the cache. We do this with a `before_invoke` callback that prepends a unique random string to the system prompt before each request. This changes the prefix, guaranteeing a cache miss.

The callback modifies the payload in-place (as required by the LLMeter callback contract).

In [6]:
from uuid import uuid4

from llmeter.callbacks.base import Callback


class CacheBuster(Callback):
    """Callback that injects a random prefix into the system prompt to prevent cache hits.

    This is useful for measuring baseline latency without prompt caching,
    so you can compare it against the cached case.
    """

    async def before_invoke(self, payload: dict) -> None:
        system = payload.get("system")
        if not system:
            return
        # Find the first text block and prepend a unique string
        for block in system:
            if "text" in block:
                block["text"] = f"[session-{uuid4().hex}] " + block["text"]
                break

## Run 1: With prompt caching (baseline)

First, we prime the cache with a single call, then run the benchmark. All requests use the same system prompt prefix, so they should hit the cache after the first one.

In [7]:
# Prime the cache
_ = endpoint.invoke(payload_with_cache)
time.sleep(1)

runner_cached = Runner(
    endpoint,
    output_path=f"outputs/cache_comparison/{model_id}/cached",
)

result_cached = await runner_cached.run(
    payload=payload_with_cache,
    n_requests=30,
    clients=3,
    run_name="with-cache",
)

Total requests:   0%|          | 0/90 [00:00<?, ?it/s]

In [8]:
print(result_cached)

{
    "total_requests": 90,
    "clients": 3,
    "n_requests": 30,
    "total_test_time": 18.54739387508016,
    "model_id": "us.amazon.nova-lite-v1:0",
    "output_path": "outputs/cache_comparison/us.amazon.nova-lite-v1:0/cached/with-cache",
    "endpoint_name": "amazon bedrock",
    "provider": "bedrock",
    "run_name": "with-cache",
    "run_description": null,
    "start_time": "2026-04-16T15:37:38Z",
    "end_time": "2026-04-16T15:37:56Z",
    "failed_requests": 0,
    "failed_requests_rate": 0.0,
    "requests_per_minute": 291.14602495476805,
    "total_input_tokens": 1080,
    "total_output_tokens": 180,
    "total_cached_input_tokens": 211288,
    "average_input_tokens_per_minute": 3493.7522994572164,
    "average_output_tokens_per_minute": 582.2920499095361,
    "time_to_last_token-average": 0.2627643963619549,
    "time_to_last_token-p50": 0.2580558335175738,
    "time_to_last_token-p90": 0.5021386420703493,
    "time_to_last_token-p99": 0.6087397387891542,
    "time_to_fir

## Run 2: Without prompt caching (cache-busted)

Now we run the same workload but with the `CacheBuster` callback, which ensures every request misses the cache.

In [9]:
runner_uncached = Runner(
    endpoint,
    output_path=f"outputs/cache_comparison/{model_id}/uncached",
    callbacks=[CacheBuster()],
)

result_uncached = await runner_uncached.run(
    payload=payload_with_cache,
    n_requests=30,
    clients=3,
    run_name="without-cache",
)

Total requests:   0%|          | 0/90 [00:00<?, ?it/s]

In [10]:
print(result_uncached)

{
    "total_requests": 90,
    "clients": 3,
    "n_requests": 30,
    "total_test_time": 17.43987720797304,
    "model_id": "us.amazon.nova-lite-v1:0",
    "output_path": "outputs/cache_comparison/us.amazon.nova-lite-v1:0/uncached/without-cache",
    "endpoint_name": "amazon bedrock",
    "provider": "bedrock",
    "run_name": "without-cache",
    "run_description": null,
    "start_time": "2026-04-16T15:37:57Z",
    "end_time": "2026-04-16T15:38:14Z",
    "failed_requests": 0,
    "failed_requests_rate": 0.0,
    "requests_per_minute": 309.6352076109381,
    "total_input_tokens": 1080,
    "total_output_tokens": 183,
    "total_cached_input_tokens": 0,
    "average_input_tokens_per_minute": 3715.622491331257,
    "average_output_tokens_per_minute": 629.5915888089075,
    "time_to_last_token-average": 0.26112701249205406,
    "time_to_last_token-p50": 0.22509589605033398,
    "time_to_last_token-p90": 0.470116974634584,
    "time_to_last_token-p99": 0.6483631999033969,
    "time_to_f

## Comparing the results

Let's compare the TTFT distributions side by side.

In [11]:
import plotly.graph_objects as go
import plotly.io as pio

from llmeter.plotting import boxplot_by_dimension, histogram_by_dimension

pio.templates.default = "plotly_white"

### Summary statistics

In [12]:
dimension = "time_to_first_token"

for label, result in [
    ("With cache", result_cached),
    ("Without cache", result_uncached),
]:
    stats = result.stats
    ttft_stats = {k: v for k, v in stats.items() if dimension in k}
    print(f"\n{label}:")
    for k, v in ttft_stats.items():
        print(f"  {k}: {v:.4f}s")
    print(f"  total_cached_input_tokens: {stats.get('total_cached_input_tokens', 0)}")
    cached_avg = stats.get("num_tokens_input_cached-average")
    if cached_avg is not None:
        print(f"  num_tokens_input_cached-average: {cached_avg:.0f}")


With cache:
  time_to_first_token-average: 0.2538s
  time_to_first_token-p50: 0.2532s
  time_to_first_token-p90: 0.4882s
  time_to_first_token-p99: 0.6013s
  total_cached_input_tokens: 211288
  num_tokens_input_cached-average: 2348

Without cache:
  time_to_first_token-average: 0.2493s
  time_to_first_token-p50: 0.2182s
  time_to_first_token-p90: 0.4523s
  time_to_first_token-p99: 0.6458s
  total_cached_input_tokens: 0
  num_tokens_input_cached-average: 0


### Boxplot comparison

In [13]:
fig = go.Figure()
fig.add_trace(boxplot_by_dimension(result=result_cached, dimension=dimension))
fig.add_trace(boxplot_by_dimension(result=result_uncached, dimension=dimension))

fig.update_layout(
    title="Time to First Token: cached vs uncached",
    xaxis_title="seconds",
    legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.99),
)
fig

### Histogram comparison

In [14]:
fig = go.Figure()
fig.add_trace(histogram_by_dimension(result_cached, dimension, xbins=dict(size=0.02)))
fig.add_trace(histogram_by_dimension(result_uncached, dimension, xbins=dict(size=0.02)))

fig.update_layout(
    title="TTFT distribution: cached vs uncached",
    xaxis_title="seconds",
    barmode="overlay",
    legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.99),
)
fig.update_traces(opacity=0.7)
fig

### Cache hit verification

We can verify that the cached run actually used the cache by comparing `total_cached_input_tokens` from `Result.stats`:

In [15]:
for label, result in [("Cached run", result_cached), ("Uncached run", result_uncached)]:
    stats = result.stats
    total_cached = stats.get("total_cached_input_tokens", 0)
    total_input = stats.get("total_input_tokens", 0)
    cache_avg = stats.get("num_tokens_input_cached-average", 0) or 0
    print(
        f"{label:14s} — total_cached_input_tokens: {total_cached:>6,}, "
        f"avg cached/request: {cache_avg:,.0f}"
    )

Cached run     — total_cached_input_tokens: 211,288, avg cached/request: 2,348
Uncached run   — total_cached_input_tokens:      0, avg cached/request: 0


## Statistical comparison

Latency data is typically skewed, so we use bootstrapping to estimate confidence intervals on the median TTFT for each run, and on the difference between them. This gives us a rigorous way to quantify the caching benefit beyond just comparing point estimates.

In [16]:
import numpy as np
from scipy.stats import bootstrap

### Median TTFT for each run

In [17]:
data_cached = [v for v in result_cached.get_dimension(dimension) if v is not None]
data_uncached = [v for v in result_uncached.get_dimension(dimension) if v is not None]

In [18]:
res = bootstrap((data_cached,), np.median, confidence_level=0.95)
print(
    f"Median TTFT (with cache):\n"
    f"  {np.median(data_cached):.4f}s "
    f"({res.confidence_interval.low:.4f}, {res.confidence_interval.high:.4f})s"
)

Median TTFT (with cache):
  0.2532s (0.2075, 0.2663)s


In [19]:
res = bootstrap((data_uncached,), np.median, confidence_level=0.95)
print(
    f"Median TTFT (without cache):\n"
    f"  {np.median(data_uncached):.4f}s "
    f"({res.confidence_interval.low:.4f}, {res.confidence_interval.high:.4f})s"
)

Median TTFT (without cache):
  0.2182s (0.1871, 0.2671)s


### Difference between medians

A positive value means the uncached run is slower (as expected). If the confidence interval excludes zero, the difference is statistically significant at the 95% level.

In [20]:
def median_difference(sample_cached, sample_uncached, axis=-1):
    return np.median(sample_uncached, axis=axis) - np.median(sample_cached, axis=axis)


res = bootstrap(
    (data_cached, data_uncached),
    median_difference,
    confidence_level=0.95,
)

diff = median_difference(data_cached, data_uncached)
print(
    f"Median TTFT difference (uncached − cached):\n"
    f"  {diff:.4f}s ({res.confidence_interval.low:.4f}, {res.confidence_interval.high:.4f})s\n"
)

if res.confidence_interval.low > 0:
    pct = diff / np.median(data_uncached) * 100
    print(f"Prompt caching reduces median TTFT by ~{pct:.1f}%")
else:
    print("The difference is not statistically significant at the 95% level.")

Median TTFT difference (uncached − cached):
  -0.0350s (-0.0815, 0.0128)s

The difference is not statistically significant at the 95% level.


## Notes

- **Cache TTL**: Bedrock prompt caches have a 5-minute TTL by default (1 hour for some Claude models with `"ttl": "1h"`). If your test run takes longer than this, later requests may miss the cache.
- **Token minimum**: Nova models require at least 1024 tokens in the cached prefix. Claude models require 1024–2048 depending on the variant. If your system prompt is too short, caching won't activate.
- **Cost**: Cached tokens are charged at a reduced rate for reads, but writes may cost more than standard input tokens. See [Bedrock pricing](https://aws.amazon.com/bedrock/pricing/) for details.